In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from master import MasterParams, MusicDBPermDir
from musicdb import PanDBIO
from gate import IOStore
from collections import Counter
from utils import poolParseIO, poolMetaModIO
from pandas import Series, DataFrame
from ioutils import FileIO
from timeutils import Timestat

io  = FileIO()
mp  = MasterParams()
ios = IOStore()

# General

In [ ]:
from lib.genius import MusicDBIO
mdbio = MusicDBIO(verbose=False)
for modVal in range(1,100):
    mdbio.prd.parseAlbumData(modVal, force=True)

In [ ]:
from ioutils import FileIO
io = FileIO()
io.get(files[0].path)

In [ ]:
from lib.metalarchives import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=True)
for modVal in range(100):
    mdbioGlobal.dir.getRawArtistModValDataDir(modVal).mkDir()
    #mdbioGlobal.dir.getRawReleaseModValDataDir(modVal).mkDir()
    #mdbioGlobal.dir.getRawTrackModValDataDir(modVal).mkDir()

In [ ]:
from lib.genius import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
srcDir = mdbioGlobal.dir.getRawModValDataDir(0)
srcDir.path

In [ ]:
from lib.allmusic import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=True)
mdbioLocal = MusicDBIO(local=True, mkDirs=False)
mdbioLocal.dir.getRawDataDir().path

In [ ]:
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioGlobal.dir.getRawArtistModValDataDir(0).path

In [ ]:
def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

from lib.genius import MusicDBIO
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
globVals    = {gVal: getModGlobVal(gVal) for gVal in range(100)}
    
import tarfile
from shutil import rmtree
tmpDir = DirInfo("/Users/tgadfort/Desktop/tmp")
tmpDir.mkDir()

modVals = list(range(100))
ts = Timestat("Creating Master ModVal Files")
for n,modVal in enumerate(modVals):
    modGlobVal = getModGlobVal(modVal)
    srcDir=DirInfo(f"/Volumes/Piggy/Discog/artists-discogs/prev/{modVal}/master")
    dstDir=DirInfo(mdbioGlobal.dir.getRawMasterModValDataDir(modVal))
    for g,(gVal,globVal) in enumerate(globVals.items()):
        srcFile = srcDir.join(f"{globVal}.tar")
        if srcFile.exists():
            if True:
                if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")
                tar = tarfile.open(srcFile.path)
                masterFiles = [FileInfo(tmpDir.join(member.name)) for member in tar.getmembers()]
                if tmpDir.exists():
                    rmtree(tmpDir.path)
                tar.extractall(path=tmpDir.path)

                retval = {}
                for finfo in masterFiles:
                    artistID,masterID = finfo.basename.split("-")
                    artistID = str(artistID)
                    masterID = str(masterID)
                    if retval.get(artistID) is None:
                        retval[artistID] = {}
                        try:
                            retval[artistID][masterID] = io.get(finfo)
                        except:
                            print(f"ERROR With {finfo.str}")

                savename = dstDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
                io.save(idata=retval, ifile=savename)
                
                rmtree(tmpDir.path)
                if g % 25 == 0: print(f"Data={len(retval)}    Saved={savename.str}")
        else:
            print(f"Tarfile [{srcFile.str}] does not exist!")
            
    ts.update(n=n+1,N=len(modVals))
ts.stop()

In [ ]:
io.get("/Volumes/Piggy/Discog/artists-discogs/0/master/mv-00-gv-94.p")

In [ ]:
from lib.genius import MusicDBIO
mdbio = MusicDBIO(local=False)

def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

globVals   = {gVal: getModGlobVal(gVal) for gVal in range(100)}
ts = Timestat("Creating ModVal Files")
modVals = list(range(1,100))
for n,modVal in enumerate(modVals):
    srcDir  = mdbio.dir.getRawModValDataDir(modVal)
    dstDir  = mdbio.dir.getRawArtistModValDataDir(modVal)
    modGlobVal = getModGlobVal(modVal)
    for g,(gVal,globVal) in enumerate(globVals.items()):
        if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")
        files = srcDir.glob(f"*{globVal}{modGlobVal}.p", debug=False)
        files = [FileInfo(ifile) for ifile in files]
        retval = {}
        for finfo in files:
            try:
                retval[finfo.basename] = io.get(finfo)
            except:
                print(f"ERROR With {finfo.str}")
        savename = dstDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
        io.save(idata=retval, ifile=savename)
        if g % 25 == 0: print(f"Data={len(retval)}    Saved={savename.str}")
    ts.update(n=n+1,N=len(modVals))
ts.stop()

In [ ]:
io.get("/Volumes/Piggy/Discog/artists-genius/0/artists/mv-00-gv-00.p")

# Group Section

In [ ]:
mdbio      = MusicDBIO(local=False, mkDirs=False)
modValDir  = mdbio.dir.getRawReleaseModValDataDir(0)
modValDir.str[:-1]

In [ ]:
def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

from lib.lastfm import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=True)
db          = mdbioGlobal.db
dstDir      = DirInfo(mdbioLocal.dir.getRawDataDir())
prevDir     = DirInfo(mdbioGlobal.dir.getRawDataDir().join("prev"))
prevDir.mkDir()

ts = Timestat("Creating ModVal Files")
modVals  = {mVal: getModGlobVal(mVal) for mVal in range(10,100)}
globVals = {gVal: getModGlobVal(gVal) for gVal in range(100)}

for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    srcDir  = mdbioGlobal.dir.getRawModValDataDir(modVal)
    #srcDir  = mdbioGlobal.dir.getRawExtraModValDataDir(modVal)
    #srcDir  = DirInfo(srcDir.str[:-1])
    print(srcDir.str)
    #modValDir  = mdbioGlobal.dir.getRawTrackModValDataDir(modVal)
        
    #files      = [FileInfo(ifile) for ifile in modValDir.glob(f"*{modGlobVal}.p", debug=False)]
    for g,(gVal,globVal) in enumerate(globVals.items()):
        if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")        
        
        ## If Extra Dir
        #files = srcDir.glob(f"*{globVal}{modGlobVal}-*.p", debug=False)
        
        ## Else
        files = srcDir.glob(f"*{globVal}{modGlobVal}.p", debug=False)
        
        files = [FileInfo(ifile) for ifile in files]
        if len(files) == 0:
            if g % 25 == 0: print("")
            continue
        retval = {}
        for finfo in files:
            try:
                retval[finfo.basename] = io.get(finfo)
            except:
                print(f"ERROR With {finfo.str}")
                
        savename = dstDir.join(f"psmv-{modGlobVal}-gv-{globVal}.p") if db in ["Deezer", "LastFM"] else dstDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
        io.save(idata=retval, ifile=savename)

        for finfo in files:
            dstFile = prevDir.join(finfo.name)
            #print(finfo.path)
            #print(dstFile.path)
            #1/0
            finfo.mvFile(dstFile, debug=False)

        if g % 25 == 0: print(f"Data={len(retval)}    Saved={savename.str}")
    ts.update(n=n+1,N=len(modVals))
ts.stop()

In [ ]:
files[0].name

In [ ]:
from lib.traxsource import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=True)
for modVal in range(100):
    break
    mdbioGlobal.dir.getRawArtistModValDataDir(modVal).mkDir()
    mdbioGlobal.dir.getRawExtraModValDataDir(modVal).mkDir()

In [ ]:
from lib.traxsource import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=True)
mdbioLocal = MusicDBIO(local=True, mkDirs=False)
srcDir = DirInfo(mdbioLocal.dir.getRawDataDir())
def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

test = False
modVals  = {mVal: getModGlobVal(mVal) for mVal in range(1,100)}
for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    #dstDir  = mdbioGlobal.dir.getRawArtistModValDataDir(modVal)
    dstDir  = mdbioGlobal.dir.getRawExtraModValDataDir(modVal)
    #dstDir  = mdbioGlobal.dir.getRawTrackModValDataDir(modVal)
    #dstDir  = DirInfo(dstDir.str[:-1])
    #dstDir = mdbioGlobal.dir.getRawArtistModValDataDir(modVal)
    modGlobVal = getModGlobVal(modVal)
    files = [FileInfo(ifile) for ifile in srcDir.glob(f"mv-{modGlobVal}-gv-*.p")]
    if test is True:
        print(len(files))
    for finfo in files:
        dstFile = dstDir.join(finfo.name)
        if test is True:
            print(finfo.path)
            print(dstFile.path)
            break
        finfo.mvFile(dstFile)
    if test is True:
        break

# Spotify Is Different

In [ ]:
def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

from lib.spotify import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=True)
dstDir      = DirInfo(mdbioLocal.dir.getRawDataDir())
prevDir     = DirInfo(mdbioGlobal.dir.getRawDataDir().join("prev"))
prevDir.mkDir()

ts = Timestat("Creating ModVal Files")
modVals  = {mVal: getModGlobVal(mVal) for mVal in range(50,100)}
globVals = {gVal: getModGlobVal(gVal) for gVal in range(100)}

for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    srcDir  = mdbioGlobal.dir.getRawModValDataDir(modVal)
    #srcDir  = mdbioGlobal.dir.getRawExtraModValDataDir(modVal)
    #srcDir  = DirInfo(srcDir.str[:-1])
    #print(srcDir.str)
    #modValDir  = mdbioGlobal.dir.getRawTrackModValDataDir(modVal)
        
    gfiles = [FileInfo(ifile) for ifile in srcDir.glob("*a*.p", debug=False)]
    gfiles = {finfo: mdbioGlobal.getGlobVal(finfo.basename) for finfo in gfiles}
    for g,(gVal,globVal) in enumerate(globVals.items()):
        if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")        

        files = [finfo for finfo,finfoGlobVal in gfiles.items() if finfoGlobVal == gVal]
        if len(files) == 0:
            if g % 25 == 0: print("")
            continue
            
        retval = {}
        for finfo in files:
            try:
                retval[finfo.basename] = io.get(finfo)
            except:
                print(f"ERROR With {finfo.str}")
        savename = dstDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
        #print(savename.path)
        #io.save(idata=retval, ifile=savename)

        for finfo in files:
            dstFile = prevDir.join(finfo.name)
            #print(finfo.path)
            #print(dstFile.path)
            #1/0
            finfo.mvFile(dstFile, debug=False)

        if g % 25 == 0: print(f"Data={len(retval)}    Saved={savename.str}")
    ts.update(n=n+1,N=len(modVals))
ts.stop()

Process [Creating ModVal Files] Start    ==> Time Is 2022-07-08 09:35:11
ModVal=50   GlobVal=00   Data=38    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-50-gv-00.p
ModVal=50   GlobVal=25   Data=39    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-50-gv-25.p
ModVal=50   GlobVal=50   Data=51    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-50-gv-50.p
ModVal=50   GlobVal=75   Data=50    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-50-gv-75.p
1/50       : Process [Creating ModVal Files] Has Run For 7 Minutes and 16 Seconds.    ==> ETR Is 5 Hours and 56 Minutes
ModVal=51   GlobVal=00   Data=41    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-51-gv-00.p
ModVal=51   GlobVal=25   Data=51    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-51-gv-25.p
ModVal=51   GlobVal=50   Data=46    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-51-gv-50.p
ModVal=51   GlobVal=75   Data=64    Saved=/Users/tgadfort/Music/Discog/artists-spotify/mv-51-gv-75.p

In [ ]:
#s = Series({finfo: getSHA1(finfo.basename) for finfo in files})

In [ ]:
s.hist(bins=100)

# Merge Local Files and RawModVal Files

In [ ]:
for modVal in range(100):
    files = DirInfo(f"/Volumes/Piggy/Discog/artists-beatport/{modVal}/track").glob("mv-*-gv-*.p")
    files = [FileInfo(ifile) for ifile in files]
    dstDir = DirInfo(f"/Volumes/Piggy/Discog/artists-beatport/{modVal}/tracks")
    for finfo in files:
        dstFile = dstDir.join(finfo.name)
        finfo.mvFile(dstFile, debug=False)
            


In [ ]:
def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

from lib.beatport import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=False)
prevDir     = DirInfo(mdbioLocal.dir.getRawDataDir().join("prev"))
prevDir.mkDir()
prevDir     = prevDir.join("tracks")
prevDir.mkDir()

modVals  = {mVal: getModGlobVal(mVal) for mVal in range(100)}
globVals = {gVal: getModGlobVal(gVal) for gVal in range(100)}

ts = Timestat("Creating ModVal Files")
for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    #modValDir  = mdbio.dir.getRawModValDataDir(modVal)
    srcDir  = mdbioLocal.dir.getRawTrackModValDataDir(modVal)
    srcDir  = DirInfo(srcDir.str[:-1])
    dstDir  = mdbioGlobal.dir.getRawTrackModValDataDir(modVal)
    #files      = [FileInfo(ifile) for ifile in modValDir.glob(f"*{modGlobVal}.p", debug=False)]
    for g,(gVal,globVal) in enumerate(globVals.items()):
        if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")
        files = srcDir.glob(f"*{globVal}{modGlobVal}.p", debug=False)
        files = [FileInfo(ifile) for ifile in files]
        if g % 25 == 0: print(f"Files={len(files)}    ", end="")
        if len(files) == 0:
            if g % 25 == 0: print("")
            continue
        #gfiles = [finfo for finfo in files if finfo.basename.endswith(f"{globVal}{modGlobVal}")]
        newRetval = {}
        for finfo in files:
            try:
                newRetval[finfo.basename] = io.get(finfo)
            except:
                print(f"ERROR With {finfo.str}")
                
        dstFile = dstDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
        if dstFile.exists():
            retval = io.get(dstFile)
        else:
            raise ValueError(f"Raw ModVal File [{dstFile.str}] does not exist")
            retval = {}
            #continue
            
        if g % 25 == 0: print(f"PrevData={len(retval)}    ", end="")
        retval.update(newRetval)
        if g % 25 == 0: print(f"Data={len(retval)}    Saved={dstFile.str}")
        io.save(idata=retval, ifile=dstFile)
        
        for finfo in files:
            dstFile = prevDir.join(finfo.name)
            finfo.mvFile(dstFile, debug=False)
            
    ts.update(n=n+1,N=len(modVals))
ts.stop()

# Next Section

In [ ]:
from lib.genius import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbio = ios.get('Genius')
mdbioLocal = MusicDBIO(local=True)
globVals   = {gVal: getModGlobVal(gVal) for gVal in range(100)}
savedir    = DirInfo(DirInfo(mdbio.dir.getRawDataDir()).join("prev"))

ts = Timestat("Creating ModVal Files")
modVals = list(range(100))
for n,modVal in enumerate(modVals):
    modValDir  = mdbio.dir.getRawAlbumModValDataDir(modVal)
    modGlobVal = getModGlobVal(modVal)
    files = modValDir.glob("*.p", debug=False)
    for ifile in files:
        finfo = FileInfo(ifile)
        dstFile = savedir.join(finfo.name)
        finfo.mvFile(dstFile, debug=False)
    print(".",end="")
    if (n+1) % 10 == 0:
        print("")
    #FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistInfoFilename(modVal,ifile.stem))) for ifile in files]

In [ ]:
retval

In [ ]:
import tarfile

for modVal in range(1,100):
    modValDir = mdbio.dir.getRawMasterModValDataDir(modVal)
    modGlobVal = getModGlobVal(modVal)
    print(modGlobVal, end="\t")
    for gVal,globVal in globVals.items():
        tarname = modValDir.join(f"{globVal}.tar")
        #print(f"{tarname.str}", end=" | ")
        tar = tarfile.open(tarname.str, "w")
        files = list(modValDir.glob(f"*{globVal}{modGlobVal}-*.p", debug=False))
        for ifile in files:
            tar.add(ifile)
        tar.close()
    print("Done")

In [ ]:
mdbio.meta.make(metatype="Link")

In [ ]:
mdbio.data.getMetaLinkData(0)

In [ ]:
modValData = mdbio.data.getModValData(0)

In [ ]:
modValData['316200'].media.media['Releases'][0].get()

In [ ]:
    def getMediaGenre(self, rData):
        media = self.getMediaData(rData, {})
        retval = {mediaType: [release.year for release in mediaTypeData if isinstance(release.year,(int,str))] for mediaType,mediaTypeData in media.items()}
        return retval
    
    def getMediaYear(self, rData):
        media = self.getMediaData(rData, {})
        retval = {mediaType: [release.year for release in mediaTypeData if isinstance(release.year,(int,str))] for mediaType,mediaTypeData in media.items()}
        return retval

In [ ]:
media = modValData['586573'].media.media
from pandas import to_datetime, Timestamp
mediaFormats = {mediaType: [release.aformat for release in mediaTypeData if isinstance(release.aformat,(dict))] for mediaType,mediaTypeData in media.items()}
genres = [record.get('Genres', []) for mediaType,mediaFormatData in mediaFormats.items() for record in mediaFormatData]
genres = Series([record.text for item in genres for record in item if record is not None]).value_counts().head(3).index.to_list()
dates  = [record.get('Date') for mediaType,mediaFormatData in mediaFormats.items() for record in mediaFormatData]
dates = [to_datetime(item.replace("\n", "").strip(), errors='ignore') for item in dates if isinstance(item,str)]
years = [item.year for item in dates if isinstance(item,Timestamp)]

In [ ]:

dates = []
for item in dates:
    try:
        to_datetime

In [ ]:
aformats

In [ ]:
["Tracks"][0].get()

In [ ]:
from lib import jiosaavn
mio = jiosaavn.MusicDBIO(mkDirs=True, verbose=True)
#for modVal in range(100):
#    mio.prd.parseAlbumData(modVal)

In [ ]:
mio.meta.make(0)

In [ ]:
tmp = mio.data.getMetaGenreData(0)
#tmp["BirthDeath"].apply(lambda x: len([val for val in x if val is not None]) if isinstance(x,list) else None).max()

In [ ]:
tmp

In [ ]:
from lib.allmusic import RawDBData
rdb = RawDBData()
rData = rdb.getArtistData("/Volumes/Piggy/Discog/artists-allmusic/0/0001936600.p")

In [ ]:
rData.show()

In [ ]:
from lib.allmusic import MusicDBID
aid = MusicDBID()
aid.get("https://www.allmusic.com/artist/mn0001936600")

In [ ]:
from utils import PoolIO
pio = PoolIO("Genius")
#pio.parse(force=True)
pio.merge()
pio.meta()
pio.sum()
pio.search()

In [ ]:
mdbio = ios.get("AllMusic", verbose=True)
#mdbio.prd.parseArtistData(0, force=True)
#mdbio.prd.parseExtraData(0, force=True)
#mdbio.prd.mergeModValData(0)
#mdbio.sum.make(verbose=True)
#mdbio.search.make(verbose=True)
#mdbio.data.getSummaryAppearanceMediaData()

In [ ]:
modValArtistData = mdbio.data.getModValArtistData(0)

In [ ]:
modValArtistData['0001035600'].show()

In [ ]:
tabsData = modValArtistData.apply(getTabs)

In [ ]:
tabsData[tabsData.apply(len) == 7]

In [ ]:
tabsData['0002102800']

In [ ]:
for db,mdbio in ios.get().items():
    if db in ["Discogs", "Spotify", "LastFM"]:
        continue
    mdbio.search.make(verbose=True)

In [ ]:
txpio.data.getModValData(0)['119900'].show()

In [ ]:
from utils import PoolIO
pio = PoolIO("MyMixTapez")
#"MetalArchives", "LastFM", "AlbumOfTheYear", "AllMusic", "Traxsource"
#pio.parse(force=True)
#pio.merge()
#pio.meta()
pio.sum()
pio.search()

In [ ]:
mdbio = ios.get("Beatport")
mdbio.data.getSummaryNameData().index.value_counts().max()

In [ ]:
mdbio = MusicDBID()
s = 'https://www.beatport.com/artist/5how/424015'
mdbio.get(s)

In [ ]:
nameData[nameData.index.isin(['2','5'])]

In [ ]:
bpio.search.make(verbose=True)

In [ ]:
mioGlobal.data.getRawFilename

In [ ]:
""" MusicDBIO Utilities """

__all__ = ["moveLocalFiles"]

from fileutils import FileInfo
from master import MasterParams
from lib.beatport import MusicDBIO
#.musicdbio

def moveLocalFiles(**kwargs):
    verbose = kwargs.get('verbose')
    if verbose: print("moveLocalFiles()")
    mp        = MasterParams()
    mioLocal  = MusicDBIO(local=True,mkDirs=True,debug=True)
    mioGlobal = MusicDBIO(local=False,mkDirs=True,debug=True)
    for modVal in mp.getModVals():
        files = mioLocal.dir.getRawModValDataDir(modVal).glob("*.p")
        _ = [FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistInfoFilename(modVal,ifile.stem))) for ifile in files]

In [ ]:
from gate import MusicDBGate
gate   = MusicDBGate()
gate.getIO("RateYourMusic").search.make(verbose=True)
gate.getIO("Genius").search.make(verbose=True)
gate.getIO("Spotify").search.make(verbose=True)
gate.getIO("LastFM").search.make(verbose=True)
gate.getIO("Deezer").search.make(verbose=True)
gate.getIO("Discogs").search.make(verbose=True)
gate.getIO("MusicBrainz").search.make(verbose=True)


#  ====> Saving [21149 / 21149 / 50498] Searchable ID => Name  Data

In [ ]:
mdbdata = gate.getIO("Genius").data
mdbdata.getSearchSingleEPMediaData()

In [ ]:
mdbdata = gate.getIO("Genius").data
artistCountsData  = mdbdata.getSummaryCountsData().join(mdbdata.getSummaryNameData())
#retval = (counts["Album"] >= 2 or counts["SingleEP"] >= 2) & (counts["Name"].len() > 0)


In [ ]:
def test(counts):
    
def artistCountsData

#(artistCountsData["Name"].map(len) > 0).sum()

In [ ]:
bpio.data.getSummaryCountsData()

In [ ]:
nameData[nameData["Name"].str.contains("Various")]

In [ ]:
nameData = DataFrame(mdbio.data.getSummaryNameData()).join(mdbio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)


In [ ]:
nameData[nameData["Name"].str.len() == 0]

In [ ]:
artistModValData = mdbio.data.getModValArtistData('92-65')
albumModValData  = mdbio.data.getModValAlbumData('92-65')

In [ ]:
searchData["id"] = searchData["url"].apply(mdbio.getdbid)

In [ ]:
searchData[searchData["name"].str.contains("Beatles")].head(1)

In [ ]:
mdbio.prd.mergeModValData(65)

In [ ]:
modValData = mdbio.data.getModValData(65)

In [ ]:
modValData['3958904065'].show()

In [ ]:
from utils import PoolIO
pio = PoolIO("Genius")
#"MetalArchives", "LastFM", "AlbumOfTheYear", "AllMusic"
#pio.parse()
#pio.merge()
#pio.meta()
#pio.sum()
pio.search()

In [ ]:
""" Web Download Utils """

__all__ = ["WebIO", "WebData"]

import urllib
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError
from time import sleep

class WebData:
    def __init__(self, data, code):
        self.data = data
        self.code = code
        
    def getData(self):
        return self.data
    
    def getCode(self):
        return self.code

url='https://api.setlist.fm/rest/1.0/artist/f0bdaa19-6f9f-42f3-b392-7782b24725b5'
user_agent   = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/35.0.1916.47 Safari/537.36'
headers = {'User-Agent':user_agent}
request  = Request(url,None,headers) #The assembled request
request.get_full_url()

In [ ]:
try:
    response = urlopen(request)
except HTTPError as err:
    x = err

In [ ]:
x.msg

In [ ]:
from pandas import read_hdf
read_hdf("/Users/tgadfort/test.hd5")
#idata.to_hdf("~/test.hd5", key='test', mode='w')

In [ ]:
ifile = "/Users/tgadfort/Desktop/test.hd5"
idata = data
if hasattr(data.__class__, 'to_hdf') and callable(getattr(data.__class__, 'to_hdf')):
    print("Saving to hd5")
    idata.to_hd5(ifile, 'w')

In [ ]:
mdbio = gate.getIO("MetalArchives")
data = mdbio.data.getSummaryNameData()
from ioutils import FileIO
io = FileIO()
io.save(idata=data.to_csv, ifile="/Users/tgadfort/Desktop/names.csv")

In [ ]:
mdbio = gate.getIO("LastFM")
#mdbio.prd.mergeModValData(modVal=0, verbose=True, maxMedia=300)
#mdbio.
#mdbio.meta.make(0)
#mdbio.sum.make()
mdbio.search.make()

In [ ]:
mdbio.meta.make(0)

In [ ]:
mdbio.data.getMeta

In [ ]:
modValData = mdbio.data.getModValData(0)

In [ ]:
for k,v in modValData.iteritems():
    print(k,'\t',[(k2,len(v2)) for k2,v2 in v.media.media.items()])
#mdbio.data.getMetaCountsData(0)

In [ ]:
tmp = mdbio.data.getModValAlbumData('0-0')

In [ ]:
tmp['52762544900'].media.media['Tracks']

In [ ]:

minCount = Series([x.aformat['Counts'] for x in tmp['52762544900'].media.media['Tracks']]).astype(int).sort_values(ascending=False).head(200).min()
[item for item in tmp['52762544900'].media.media['Tracks'] if int(item.aformat['Counts']) >= minCount]

In [ ]:
minCount

In [ ]:
from lib.spotify import MusicDBIO
mdbio = MusicDBIO()
mdbio.prd.mergeModValData()

In [ ]:
from lib.rateyourmusic import moveLocalFiles, removeLocalFiles
moveLocalFiles()
removeLocalFiles()

In [ ]:
pio = PoolIO("Spotify")

In [ ]:
from lib.metalarchives import moveLocalFiles
#moveLocalFiles()
from time import sleep
from utils import PoolIO
pio = PoolIO("LastFM")
#pio.parse()
pio.merge(verbose=True, maxMedia=300)
pio.meta()
pio.sum()

# DB-Specific

In [ ]:
from lib.discogs import moveLocalFiles
moveLocalFiles()

In [ ]:
from utils import PoolIO
pio = PoolIO("Discogs")
pio.parse()
#pio.meta()
#pio.sum()

# Deezer

In [ ]:
mdbio = gate.getIO("Deezer")
artistRelatedData = mdbio.data.getRelatedArtistsData()
artistRelatedData.name = "RelatedArtists"
artistRelatedData = DataFrame(artistRelatedData)
artistInfoData    = mdbio.data.getArtistsInfoData()

In [ ]:
deezerData = artistInfoData.join(artistRelatedData)

In [ ]:
modValData = mdbio.data.getModValData(13)

In [ ]:
modValData.loc['13'].show()

In [ ]:
row = deezerData[deezerData.index == '13']
row

In [ ]:
if '13x' in deezerData.index:
    print(deezerData.loc['13', 'picture'])

In [ ]:

#generalData = {"Type": artistType}
#extraData   = {"TopTracks": artistIDData["Artist"].tracks}
if '13' in deezerData.index:
    profile = modValData.loc['13'].profile
    extraData = getattr(profile, 'extra') if hasattr(profile, 'extra') else {}
    if isinstance(extraData,dict):
        extraData['Image']   = deezerData.loc['13', 'picture']
        extraData['Albums']  = deezerData.loc['13', 'albums']
        extraData['Fans']    = deezerData.loc['13', 'fans']
        extraData['Related'] = deezerData.loc['13', 'RelatedArtists']
    print(extraData)

In [ ]:
todo = ["MetalArchives", "AllMusic"]
from utils import PoolIO
pio = PoolIO("Genius")
pio.parse(force=True)
mdbios["Genius"].prd.mergeModValData()
pio.meta()
pio.sum()

In [ ]:
extData = {}
for modVal in range(100):    
    print(modVal)
    extData.update({artistID: artistData.profile.external for artistID,artistData in mdbio.data.getModValArtistData(modVal).iteritems()})
    
#Series(extData).apply(Series)
deezerData = Series({artistID: dbData.get('Deezer') for artistID,dbData in extData.items() if isinstance(dbData,dict)}, name="Deezer")
geniusData = Series({artistID: dbData.get('Genius') for artistID,dbData in extData.items() if isinstance(dbData,dict)}, name="Genius")

In [ ]:
dfGenius = DataFrame(geniusData)
dfGenius["Num"] = dfGenius["Genius"].apply(lambda x: len(x) if isinstance(x,list) else 0)
dfDeezer = DataFrame(deezerData)
dfDeezer["Num"] = dfDeezer["Deezer"].apply(lambda x: len(x) if isinstance(x,list) else 0)

In [ ]:
geniusMergeData = dfGenius[dfGenius["Num"] == 1]["Genius"].apply(lambda x: x[0].get())
geniusMergeDataDF = geniusMergeData.apply(Series)

In [ ]:
deezerMergeData = dfDeezer[dfDeezer["Num"] == 1]["Deezer"].apply(lambda x: x[0].get())
deezerMergeDataDF = deezerMergeData.apply(Series)


In [ ]:
mdbios["Deezer"].getdbid(" https://www.deezer.com/artist/495")

In [ ]:
deezerLookup = deezerMergeDataDF['href'].apply(lambda x: x.split("/")[-1]).to_dict()

In [ ]:
refData = mdbios["Genius"].data.getSummaryRefData()
refData.name = "href"
refData = DataFrame(refData)

In [ ]:
geniusMergeDataDF.index.name = "MusicBrainz"
geniusMergeDataDF = geniusMergeDataDF.reset_index()
lookup = geniusMergeDataDF["MusicBrainz"]
lookup.index = geniusMergeDataDF['href']
lookup

In [ ]:
refData.index.name = "GeniusID"
refData = refData.reset_index()
refLookup = refData['GeniusID']
refLookup.index = refData["href"]

In [ ]:
tmp = DataFrame(lookup).join(DataFrame(refLookup))
tmp = tmp[tmp["GeniusID"].notna()]

In [ ]:
glookup = dict(zip(tmp["MusicBrainz"],tmp["GeniusID"]))

In [ ]:
from musicdb import MusicDBIgnoreData, MusicDBIO
pdbio = MusicDBIO()
mmeDF = pdbio.getData()
deezerMatches = mmeDF["MusicBrainz"].apply(deezerLookup.get)
deezerMatches.name = "Deezer"

In [ ]:
mmeDF["Genius"].count()

In [ ]:
mmeDF = mmeDF.join(deezerMatches)

In [ ]:
deezerMatches

In [ ]:
pdbio = MusicDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF

In [ ]:
pdbio.saveData(mmeDF)

In [ ]:

################################################################################################################
# Genius
################################################################################################################
from hashlib import md5
class utilsGenius:
    def __init__(self):
        self.baseURL = "https://genius.com/artists/"
        self.relURL  = "/artists/"
        
    def getArtistID(self, href, debug=False):
        if href.startswith(self.baseURL):
            if debug:
                print("Removing {0} from url --> {1}".format(self.baseURL,href))
            href = href[len(self.baseURL):]        
        if href.startswith(self.relURL):
            if debug:
                print("Removing {0} from url --> {1}".format(self.relURL,href))
            href = href[len(self.relURL):]
        name = href.split("/+albums")[0]
        if name.endswith("/"):
            name = href[:-1]
        if debug:
            print("Raw URL: {0}".format(name))
        #name = self.quoteIt(name, debug=debug)
        if debug:
            print("Raw URL (Post Quote): {0}".format(name))
        if name is None:
            return None
        name = "{0}{1}".format(self.baseURL,name)
        if debug:
            print("Full URL: {0}".format(name))
        name = name.lower()
        if debug:
            print("Final URL: {0}".format(name))
        m = md5()
        for val in name.split(" "):
            m.update(val.encode('utf-8'))
        hashval = m.hexdigest()
        artistID  = str(int(hashval, 16) % int(1e8))
        return artistID

In [ ]:
refData = mdbios["Genius"].data.getSummaryRefData()

In [ ]:
util = utilsGenius()
oldGenLookup = refData.apply(util.getArtistID)

In [ ]:
oldGenLookup = {v: k for k,v in oldGenLookup.to_dict().items()}
mmeDF = pdbio.getData()

In [ ]:
genvals = mmeDF["GeniusOld"].apply(oldGenLookup.get)

In [ ]:
mbgen = mmeDF["Genius"].to_dict()